<a href="https://colab.research.google.com/github/Diagnost13/simulative-git/blob/master/%D0%9A%D0%B5%D0%B9%D1%81_%D0%9B%D0%BE%D0%B3%D0%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **1 кейс**

**Работа с логами**

**Важно**

Перед началом решения выполните следующую ячейку, чтобы загрузить необходимый для работы файл.

In [2]:
!wget https://gist.github.com/Vs8th/38d5d914171c84166728a9746d212bad/raw/auto_purchase.log

--2026-03-12 08:15:58--  https://gist.github.com/Vs8th/38d5d914171c84166728a9746d212bad/raw/auto_purchase.log
Resolving gist.github.com (gist.github.com)... 140.82.114.4
Connecting to gist.github.com (gist.github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/38d5d914171c84166728a9746d212bad/raw/auto_purchase.log [following]
--2026-03-12 08:15:58--  https://gist.githubusercontent.com/Vs8th/38d5d914171c84166728a9746d212bad/raw/auto_purchase.log
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 459418 (449K) [text/plain]
Saving to: ‘auto_purchase.log’

auto_purchase.log   100%[===================>] 448.65K  --.-KB/s    in 0.04s   

2026-03-12 08:15:59 (11.9

Чтобы посмотреть как он выглядит выполните следующую ячейку.

In [3]:
with open('auto_purchase.log', 'r') as f:
    lines = f.readlines()

for line in lines[400:550]:
    print(line)

INFO  | 2022-11-26 00:01:03,024  | file: demon_auto_purchase_bundle.py |  line: 54 | [demon] Cегодня 2022-11-26, количество людей с автопродлением подписки: 0

INFO  | 2022-11-26 00:01:03,027  | file: demon_auto_purchase_bundle.py |  line: 57 | [demon] Выход из программы

INFO  | 2022-11-27 00:01:02,455  | file: demon_auto_purchase_bundle.py |  line: 44 | [demon] Поверяем истекающие премиум-пакеты

INFO  | 2022-11-27 00:01:02,480  | file: demon_auto_purchase_bundle.py |  line: 54 | [demon] Cегодня 2022-11-27, количество людей с автопродлением подписки: 0

INFO  | 2022-11-27 00:01:02,483  | file: demon_auto_purchase_bundle.py |  line: 57 | [demon] Выход из программы

INFO  | 2022-11-28 00:01:02,406  | file: demon_auto_purchase_bundle.py |  line: 44 | [demon] Поверяем истекающие премиум-пакеты

INFO  | 2022-11-28 00:01:02,424  | file: demon_auto_purchase_bundle.py |  line: 54 | [demon] Cегодня 2022-11-28, количество людей с автопродлением подписки: 0

INFO  | 2022-11-28 00:01:02,427  | f

### **Решения**

#### **Задача 1**

Ваша задача написать функцию `count_success_and_failure`, которая принимает на вход путь к файлу с логами и подсчитывает количество успешных продлений и ошибок при списании. Функция должна вернуть кортеж из двух значений: количества успешных попыток и неуспешных.

**Решение**

Напишите свое решение ниже

In [4]:

def count_success_and_failure(file_path):

    success_count = 0
    failure_count = 0
    last_user_id = None

    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            # Ищем начало попытки обновления подписки
            if 'Обновляем подписку пользователю id:' in line:
                # Извлекаем ID пользователя
                parts = line.split('id:')
                if len(parts) > 1:
                    last_user_id = parts[1].split(',')[0].strip()
                    success_count += 1  # Предполагаем успех, пока не доказано обратное

            # Ищем ошибки для текущего пользователя
            elif 'ERROR' in line and last_user_id is not None:
                if f'У пользователя с id: {last_user_id}' in line:
                    failure_count += 1
                    success_count -= 1  # Отменяем предыдущий предполагаемый успех
                    last_user_id = None  # Сбрасываем ID после обработки ошибки

    return (success_count, failure_count)

count_success_and_failure('auto_purchase.log')

(1034, 186)

✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [5]:
res = count_success_and_failure('auto_purchase.log')

try:
    assert res == (1034, 186)
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


#### **Задача 2**

Ваша задача написать функцию `auto_renewal_sub`, которая принимает на вход путь к файлу с логами и обрабатывает количество клиентов с автопродлением подписки. Мы хотим посмотреть на изменение этого показателя в динамике: посчитайте сглаженные значения с помощью метода скользящего среднего и метода медианного сглаживания.  

**Примечание:** При сглаживании берем все предыдущие значения, включая текущее, будущие значения не берем. Если в один день наблюдаем несколько записей об автопродлении - берем максимальное из имеющихся число клиентов с подпиской.

Функция должна записать в файл `auto_renewal_sub.txt` два списка, предварив их соответствущими обозначениями:

`Среднее: [2.0, 1.0, 0.67...]`

`Медиана: [2, 2, 0...]`

**Решение**

Напишите свое решение ниже

In [6]:
from typing import ValuesView
import datetime
import statistics

def auto_renewal_sub(log_file_path):
    # Считываем данные из файла
    logs = []
    with open(log_file_path, 'r') as f:
        logs.extend(f.readlines())
    parts = [[part.strip() for part in log.split('|')] for log in logs]

    # Извлекаем информацию из логов
    data = [{'timestamp': datetime.datetime.strptime(part[1], '%Y-%m-%d %H:%M:%S,%f'),
              'message': part[4].replace("[demon] ", "").strip(),
              'system': part[0]}
            for part in parts]

    # Создаем пустой словарь для хранения данных
    result = {}

    # Проходим по всем сообщениям из списка
    for item in data:
        # Извлекаем дату из timestamp
        date = item['timestamp'].date()

        # Если сообщение содержит информацию о количестве людей с автопродлением подписки,
        # то обновляем соответствующую запись в словаре, если новое значение больше
        if 'количество людей с автопродлением подписки' in item['message']:
            count = int(item['message'].split(': ')[-1])
            if date in result:
                result[date] = max(result[date], count)  # Берем максимум
            else:
                result[date] = count

    def moving_average_median(values):
        # Проверяем корректность размера списка.
        if len(values) < 1:
            raise ValueError("Размер списка должен быть не меньше 1.")

        # Вычисляем скользящее среднее и медиану для каждого значения.
        ma = list(map(lambda i: round(sum(values[:i+1])/len(values[:i+1]), 2), range(len(values))))
        med = [int(statistics.median(values[:i+1])) for i in range(len(values))]

        return (ma, med)

    values = list(result.values())
    ma, med = moving_average_median(values)

    # Записываем результат в файл
    with open('auto_renewal_sub.txt', 'w') as f:
        f.write("Среднее: {}\n".format(ma))
        f.write("Медиана: {}\n".format(med))

auto_renewal_sub('auto_purchase.log')

In [7]:
with open('auto_renewal_sub.txt', 'r', encoding='utf-8') as file:
    content = file.read()
    print(content)

Среднее: [1.0, 0.5, 0.33, 0.25, 0.2, 0.17, 0.14, 0.12, 0.11, 0.1, 0.09, 0.08, 0.08, 0.07, 0.07, 0.06, 0.06, 0.06, 0.05, 0.05, 0.05, 0.05, 0.04, 0.04, 0.04, 0.04, 0.04, 0.04, 0.03, 0.03, 0.06, 0.09, 0.09, 0.09, 0.09, 0.08, 0.08, 0.08, 0.08, 0.07, 0.07, 0.07, 0.12, 0.14, 0.13, 0.13, 0.13, 0.15, 0.16, 0.16, 0.16, 0.15, 0.15, 0.15, 0.15, 0.14, 0.16, 0.17, 0.17, 0.17, 0.2, 0.21, 0.21, 0.2, 0.22, 0.21, 0.21, 0.21, 0.2, 0.2, 0.2, 0.19, 0.21, 0.2, 0.2, 0.2, 0.19, 0.21, 0.22, 0.21, 0.22, 0.22, 0.22, 0.21, 0.21, 0.21, 0.22, 0.22, 0.21, 0.22, 0.24, 0.24, 0.25, 0.24, 0.24, 0.25, 0.25, 0.24, 0.24, 0.25, 0.25, 0.25, 0.24, 0.24, 0.24, 0.24, 0.23, 0.23, 0.23, 0.23, 0.23, 0.23, 0.23, 0.23, 0.23, 0.23, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.24, 0.25, 0.24, 0.24, 0.24, 0.25, 0.25, 0.25, 0.25, 0.25, 0.24, 0.25, 0.27, 0.27, 0.27, 0.26, 0.27, 0.28, 0.28, 0.28, 0.28, 0.29, 0.29, 0.3, 0.3, 0.3, 0.3, 0.29, 0.29, 0.3, 0.31, 0.31, 0.31, 0.31, 0.31, 0.33, 0.33, 0.34, 0.34, 0.34,

✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [8]:
# Здесь будет скачиваться файл с эталонным ответом

!wget https://gist.github.com/Vs8th/846cbc97a722ab825cda4dda3b2b3434/raw/cor_auto_renewal.txt

import pandas as pd

user_answer = pd.read_csv('auto_renewal_sub.txt')
correct_answer = pd.read_csv('cor_auto_renewal.txt')

--2026-03-12 08:16:26--  https://gist.github.com/Vs8th/846cbc97a722ab825cda4dda3b2b3434/raw/cor_auto_renewal.txt
Resolving gist.github.com (gist.github.com)... 140.82.114.4
Connecting to gist.github.com (gist.github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/846cbc97a722ab825cda4dda3b2b3434/raw/cor_auto_renewal.txt [following]
--2026-03-12 08:16:26--  https://gist.githubusercontent.com/Vs8th/846cbc97a722ab825cda4dda3b2b3434/raw/cor_auto_renewal.txt
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2431 (2.4K) [text/plain]
Saving to: ‘cor_auto_renewal.txt’

cor_auto_renewal.tx 100%[===================>]   2.37K  --.-KB/s    in 0s      

2026-03-12 08:1

In [9]:
try:
  assert (user_answer == correct_answer).all().all(), 'Ответы не совпадают'
except Exception as err:
  raise AssertionError(f'При проверке возникла ошибка {repr(err)}')
else:
  print('Поздравляем, Вы справились и успешно прошли все проверки!!')

Поздравляем, Вы справились и успешно прошли все проверки!!


#### **Задача 3**

Напишите функцию `sub_renewal_by_day`, которая принимает на вход путь к файлу с логами и анализирует взаимосвязь дня продления подписки и количества продлений в этот день. Функция должна записать в файл `weekdays.txt` аналитическую записку в формате:

**`Количество обновлений подписки по дням недели:`**

**`Понедельник: 6`**

**`Вторник: 7`**

**`Среда: 8`**

**`...`**

**Решение**

Напишите свое решение ниже

In [10]:
def sub_renewal_by_day(file_path):
    # Считываем данные из файла
    logs = []
    with open(file_path, 'r') as f:
        logs.extend(f.readlines())
    parts = [[part.strip() for part in log.split('|')] for log in logs]

    # Извлекаем информацию из логов
    data = [{'timestamp': datetime.datetime.strptime(part[1], '%Y-%m-%d %H:%M:%S,%f'),
             'message': part[4].replace("[demon] ", "").strip(),
             'system': part[0]}
            for part in parts]

    # Проверяем количество ошибок при списании к успешным продлениям
    success_count = [data[i] for i in range(len(data))
                     if 'Обновляем подписку пользователю id:' in data[i]['message'] and
                        (i == len(data)-1 or 'ошибка при списании' not in data[i+1]['message'])]

    failure_count = [data[i+1] for i in range(len(data)-1)
                     if 'ошибка при списании' in data[i+1]['message'] and
                        'Обновляем подписку пользователю id:' in data[i]['message']]

    # 2. Группируем данные по дням недели и времени суток обновления подписки
    day_of_week_count = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0}
    for d in success_count:
        day_of_week_count[d['timestamp'].weekday()] += 1

    weekdays = {
        0: 'Понедельник',
        1: 'Вторник',
        2: 'Среда',
        3: 'Четверг',
        4: 'Пятница',
        5: 'Суббота',
        6: 'Воскресенье'
    }

    with open('weekdays.txt', 'w') as f:
        print('Количество обновлений подписки по дням недели:', file=f)
        for day, count in day_of_week_count.items():
            print(f'{weekdays[day]}: {count}', file=f)

sub_renewal_by_day('auto_purchase.log')


In [11]:

def sub_renewal_by_day(file_path):

    ...

sub_renewal_by_day('auto_purchase.log')

✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [12]:
# Здесь будет скачиваться файл с эталонным ответом

!wget https://gist.github.com/Vs8th/c4c382f50761b5b9e64795eb89d49fda/raw/cor_weekdays.txt

import pandas as pd

user_answer = pd.read_csv('weekdays.txt')
correct_answer = pd.read_csv('cor_weekdays.txt')

--2026-03-12 08:18:14--  https://gist.github.com/Vs8th/c4c382f50761b5b9e64795eb89d49fda/raw/cor_weekdays.txt
Resolving gist.github.com (gist.github.com)... 140.82.114.4
Connecting to gist.github.com (gist.github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/c4c382f50761b5b9e64795eb89d49fda/raw/cor_weekdays.txt [following]
--2026-03-12 08:18:14--  https://gist.githubusercontent.com/Vs8th/c4c382f50761b5b9e64795eb89d49fda/raw/cor_weekdays.txt
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 238 [text/plain]
Saving to: ‘cor_weekdays.txt’

cor_weekdays.txt    100%[===================>]     238  --.-KB/s    in 0s      

2026-03-12 08:18:14 (5.43 MB/s) - ‘cor_

In [13]:
try:
  assert (user_answer == correct_answer).all().all(), 'Ответы не совпадают'
except Exception as err:
  raise AssertionError(f'При проверке возникла ошибка {repr(err)}')
else:
  print('Поздравляем, Вы справились и успешно прошли все проверки!!')

Поздравляем, Вы справились и успешно прошли все проверки!!
